In [79]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (
    SparkSession.builder
    .appName("PagilaLearning") # Job name
    .master("local[*]") # local running (not cluster)
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.8" # postgres driver
    )
    .getOrCreate() # Starts the engine and creates a SparkSession object
)

In [122]:
jdbc_url = "jdbc:postgresql://localhost:5433/pagila"
properties = {
      "user": "postgres",
      "password": "123456",
      "driver": "org.postgresql.Driver"
  }

- Output the number of movies in each category, sorted in descending order.
- Output the 10 actors whose movies rented the most, sorted in descending order.
- Output the category of movies on which the most money was spent.
- Output the names of movies that are not in the inventory.
- Output the top 3 actors who have appeared most in movies in the “Children” category. If several actors have the same number of movies, output all of them.
- Output cities with the number of active and inactive customers (active - customer.active = 1). Sort by the number of inactive customers in descending order.
- Output the category of movies that have the highest number of total rental hours in the cities (customer.address_id in this city), and that start with the letter “a”. Do the same for cities with a “-” symbol.

In [95]:
def read_tables(*table_names):
    tables = {}
    for table_name in table_names:
        tables[table_name] = spark.read.jdbc(
            url=jdbc_url,
            table=f"public.{table_name}",
            properties=properties
        )
    return tables

In [101]:
tables = read_tables("film_category", "category", "film")
film_category = tables["film_category"]
category = tables["category"]
film = tables["film"]
film_category.join(category, film_category.category_id == category.category_id, "inner")\
.groupBy("name")\
.agg(F.count(film_category.film_id).alias("film_count"))\
.orderBy(F.desc("film_count"))\
.show()

+-----------+----------+
|       name|film_count|
+-----------+----------+
|      Drama|       152|
|      Music|       152|
|     Travel|       151|
|    Foreign|       150|
|      Games|       150|
|   Children|       150|
|     Action|       149|
|     Sci-Fi|       149|
|  Animation|       148|
|     Family|       147|
|   Classics|       147|
|        New|       147|
|     Sports|       145|
|Documentary|       145|
|     Comedy|       143|
|     Horror|       142|
+-----------+----------+



In [102]:
tables = read_tables("film_actor", "actor")
film_actor, actors = tables["film_actor"], tables["actor"]

film_actor.join(actors, film_actor.actor_id == actors.actor_id, "inner")\
.groupBy("first_name", "last_name")\
.agg(F.count(film_actor.film_id).alias("film_count"))\
.orderBy(F.desc("film_count"))\
.show(10)

+----------+-----------+----------+
|first_name|  last_name|film_count|
+----------+-----------+----------+
|     SUSAN|      DAVIS|        54|
|      GINA|  DEGENERES|        42|
|    WALTER|       TORN|        41|
|      MARY|     KEITEL|        40|
|   MATTHEW|     CARREY|        39|
|    SANDRA|     KILMER|        37|
|  SCARLETT|      DAMON|        36|
|       UMA|       WOOD|        35|
|   GROUCHO|      DUNST|        35|
|    ANGELA|WITHERSPOON|        35|
+----------+-----------+----------+
only showing top 10 rows


In [104]:
tables = read_tables("inventory", "rental", "payment", "film_category", "category")
inventory, rental, payment, film_category, category = tables["inventory"], tables["rental"], tables["payment"], tables["film_category"], tables["category"]

payment.join(rental, payment.rental_id == rental.rental_id, "inner")\
.join(inventory, rental.inventory_id == inventory.inventory_id, "inner")\
.join(film_category, inventory.film_id == film_category.film_id, "inner")\
.join(category, film_category.category_id == category.category_id, "inner")\
.groupBy("name")\
.agg(F.sum(payment.amount).alias("total_spent"))\
.orderBy(F.desc("total_spent"))\
.show(1)

+------+-----------+
|  name|total_spent|
+------+-----------+
|Action|   26505.44|
+------+-----------+
only showing top 1 row


In [105]:
tables = read_tables("film", "inventory")
film, inventory = tables["film"], tables["inventory"]
film.join(inventory, film.film_id == inventory.film_id, "left_anti")\
.select("title")\
.show()

+--------------------+
|               title|
+--------------------+
|      CHOCOLATE DUCK|
|       BUTCH PANTHER|
|        VOLUME HOUSE|
|      ORDER BETRAYED|
|        TADPOLE PARK|
|    KILL BROTHERHOOD|
|FRANKENSTEIN STRA...|
|    CROSSING DIVORCE|
|    SUICIDES SILENCE|
|       CATCH AMISTAD|
|     PERDITION FARGO|
|       FLOATS GARDEN|
|           GUMP DATE|
|        WALLS ARTIST|
|  GLADIATOR WESTWARD|
|         HOCUS FRIDA|
|ARSENIC INDEPENDENCE|
|         MUPPET MILE|
|   FIREHOUSE VIETNAM|
|       ROOF CHAMPION|
+--------------------+
only showing top 20 rows


In [106]:
tables = read_tables("film_category", "category", "film_actor", "actor")
film_category, category, film_actor, actors = tables["film_category"], tables["category"], tables["film_actor"], tables["actor"]
film_category.join(category, film_category.category_id == category.category_id, "inner")\
.join(film_actor, film_category.film_id == film_actor.film_id, "inner")\
.join(actors, film_actor.actor_id == actors.actor_id, "inner")\
.filter(category.name == "Children")\
.groupBy("first_name", "last_name")\
.agg(F.count(film_actor.film_id).alias("film_count"))\
.orderBy(F.desc("film_count"))\
.show(3)

+----------+---------+----------+
|first_name|last_name|film_count|
+----------+---------+----------+
|    SIDNEY|    CROWE|         9|
|      EWAN|  GOODING|         9|
|   RICHARD|     PENN|         9|
+----------+---------+----------+
only showing top 3 rows


In [107]:
tables = read_tables("customer", "address", "city")
customer, address, city = tables["customer"], tables["address"], tables["city"]
customer.join(address, customer.address_id == address.address_id, "inner")\
.join(city, address.city_id == city.city_id, "inner")\
.groupBy("city")\
.agg(F.sum(F.when(customer.active == 1, 1).otherwise(0)).alias("active_customers"),
     F.sum(F.when(customer.active == 0, 1).otherwise(0)).alias("inactive_customers"))\
.orderBy(F.desc("inactive_customers"))\
.show()

+--------------------+----------------+------------------+
|                city|active_customers|inactive_customers|
+--------------------+----------------+------------------+
|San Juan Bautista...|             383|                18|
|            Uluberia|               0|                 1|
|           Najafabad|               0|                 1|
|           Pingxiang|               0|                 1|
|            Xiangfan|               0|                 1|
|          Kumbakonam|               0|                 1|
|         Szkesfehrvr|               0|                 1|
|    Charlotte Amalie|               0|                 1|
|              Kamyin|               0|                 1|
|              Daxian|               0|                 1|
|       Coatzacoalcos|               0|                 1|
|             Wroclaw|               0|                 1|
|              Ktahya|               0|                 1|
|              Amroha|               0|                 

In [121]:
tables = read_tables(
    "customer",
    "address",
    "city",
    "inventory",
    "film_category",
    "category",
    "rental"
)

customer = tables["customer"]
address = tables["address"]
city = tables["city"]
inventory = tables["inventory"]
film_category = tables["film_category"]
category = tables["category"]
rental = tables["rental"]


rental_duration = (
    rental
    .filter(F.col("return_date").isNotNull())
    .withColumn(
        "rental_duration_hours",
        (
            F.col("return_date").cast("long")
            - F.col("rental_date").cast("long")
        ) / 3600
    )
)


result = (
    customer
    .join(address, "address_id")
    .join(city, "city_id")
    .join(rental_duration, "customer_id")
    .join(inventory, "inventory_id")
    .join(film_category, "film_id")
    .join(category, "category_id")

    .filter(
        F.lower(F.col("city")).startswith("a")
        | F.col("city").contains("-")
    ).groupBy("name")

    .agg(
        F.sum("rental_duration_hours")
        .alias("total_rental_hours")
    )

    .orderBy(F.desc("total_rental_hours"))
)

result.show(1, truncate=False)

+----+------------------+
|name|total_rental_hours|
+----+------------------+
|New |88538.24611111115 |
+----+------------------+
only showing top 1 row
